# Sub-ImageNet Training in Google Colab

This notebook runs any of the Sub-ImageNet training scripts in this folder. It also downloads the pretrained backbone weights expected by the original scripts.

In [ ]:
# Optional: mount Google Drive if your dataset or checkpoints live there.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
    print(f'Cloned repository to {repo_dir}')
else:
    print(f'Repository already present at {repo_dir}')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/DVBW')
WORKDIR = PROJECT_DIR / 'Sub-ImageNet'
DATA_DIR = WORKDIR / 'data' / 'sub-imagenet-200'

if not WORKDIR.exists():
    raise FileNotFoundError(
        f'Expected the repository at {PROJECT_DIR}. Upload or clone the repo first, then rerun this cell.'
    )

os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')
print(f'Dataset directory: {DATA_DIR}')

In [ ]:
%pip install -q matplotlib tqdm pillow numpy

In [ ]:
import urllib.request

checkpoint_dir = WORKDIR / 'checkpoint'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

weights = {
    'resnet18-f37072fd.pth': 'https://download.pytorch.org/models/resnet18-f37072fd.pth',
    'vgg19_bn-c79401a0.pth': 'https://download.pytorch.org/models/vgg19_bn-c79401a0.pth',
}

for filename, url in weights.items():
    destination = checkpoint_dir / filename
    if destination.exists():
        print(f'Found {destination.name}')
    else:
        print(f'Downloading {destination.name}...')
        urllib.request.urlretrieve(url, destination)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Sub-ImageNet dataset not found at {DATA_DIR}. Put your dataset there or update DATA_DIR in the cell above.'
    )

In [ ]:
SCRIPT_NAME = 'train_standard.py'

PRESETS = {
    'train_standard.py': [
        '--gpu-id', '0',
        '--checkpoint', './checkpoint/benign/resnet_colab',
        '--data_dir', str(DATA_DIR)
    ],
    'train_standard_vgg.py': [
        '--gpu-id', '0',
        '--checkpoint', './checkpoint/benign/vgg_colab',
        '--data_dir', str(DATA_DIR)
    ],
    'train_watermarked.py': [
        '--gpu-id', '0',
        '--poison-rate', '0.1',
        '--checkpoint', './checkpoint/infected/resnet_badnets_cross_colab',
        '--trigger', './triggers/Trigger_cross.png',
        '--alpha', './triggers/Alpha_cross.png',
        '--y-target', '0',
        '--data_dir', str(DATA_DIR)
    ],
    'train_watermarked_vgg.py': [
        '--gpu-id', '0',
        '--poison-rate', '0.1',
        '--checkpoint', './checkpoint/infected/vgg_badnets_cross_colab',
        '--trigger', './triggers/Trigger_cross.png',
        '--alpha', './triggers/Alpha_cross.png',
        '--y-target', '0',
        '--data_dir', str(DATA_DIR)
    ]
}

# Add or override arguments here. Example: EXTRA_ARGS = ['--epochs', '10', '--num_class', '200']
EXTRA_ARGS = []

print('Selected script:', SCRIPT_NAME)
print('Base args:', PRESETS[SCRIPT_NAME])
print('Extra args:', EXTRA_ARGS)

In [ ]:
import shutil

print('GPU available:', shutil.which('nvidia-smi') is not None)
if shutil.which('nvidia-smi') is not None:
    !nvidia-smi

In [ ]:
import subprocess
import sys

command = [sys.executable, SCRIPT_NAME, *PRESETS[SCRIPT_NAME], *EXTRA_ARGS]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)